[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/077.%20The%20Dynamic%20%26%20Real-Time%20VRP/P77-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 77. The Dynamic & Real-Time VRP
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Transportation network represented as graph G = (V, E)
- Dynamic requests arrive according to non-homogeneous Poisson process
- Vehicles have capacity constraints and time windows
- System state evolves continuously over time horizon T
- Travel times and costs are deterministic but time-varying

### Approach (step-by-step)
1. **Model the dynamic system as a continuous-time Markov Decision Process**
2. **Define the state space with vehicle positions, routes, requests, and constraints**
3. **Formulate the dynamic optimization objective with multiple competing goals**
4. **Implement time-evolving constraints for capacity, time windows, and route continuity**
5. **Solve using mathematical programming for small instances with exhaustive search**

### What to look for in the results
- Optimal vehicle assignments for dynamic requests
- Trade-offs between travel cost, waiting time, and route modifications
- Impact of time windows on system performance
- Computational complexity as problem size increases

### Concrete example (from the source)
Three-vehicle dynamic scenario in 5×5 grid city:
- Vehicle 1: Position (1,1), route to (2,3) and (4,4)
- Vehicle 2: Position (3,2), route to (5,5)
- Vehicle 3: Position (2,4), returning to depot (1,1)
- New requests at 9:15 AM with specific time windows

### Visualization(s)
- Vehicle positions and routes over time
- Request arrival patterns and service assignments
- Cost breakdown and system performance metrics

### Why this Tier exists vs earlier Tiers
This is Tier 1, establishing the mathematical foundation for dynamic vehicle routing. Unlike static VRP formulations, this tier handles:
- Real-time request arrivals
- Continuous time evolution
- Dynamic reassignment decisions
- Multi-objective optimization under uncertainty

### Pros / Cons vs Tier 1..(k-1)
**Pros:**
- Provides exact optimal solutions for small instances
- Establishes theoretical performance bounds
- Clear mathematical formulation and assumptions
- Comprehensive sensitivity analysis capabilities

**Cons:**
- Computationally intractable for large instances
- Requires complete information about future arrivals
- Assumes deterministic travel times
- Limited scalability for real-time applications

### When to use this Tier
- Small-scale dynamic routing problems (≤ 5 vehicles, ≤ 15 requests)
- Benchmarking heuristic and metaheuristic methods
- Academic research and theoretical analysis
- Problems requiring provable optimality guarantees

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for mathematical formulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import permutations, combinations
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import time
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Request:
    """Dynamic service request with time windows and priority"""
    id: int
    origin: Tuple[int, int]  # (x, y) coordinates
    destination: Tuple[int, int]
    demand: int
    arrival_time: float  # When request arrives in system
    time_window_early: float  # Earliest service time
    time_window_late: float   # Latest service time
    priority: int  # 1=high, 2=medium, 3=low

@dataclass
class Vehicle:
    """Vehicle with position, capacity, and current route"""
    id: int
    position: Tuple[int, int]
    capacity: int
    remaining_capacity: int
    planned_route: List[Tuple[int, int]]  # List of positions to visit
    current_time: float = 0.0

@dataclass
class DVRPState:
    """Complete state of dynamic VRP at time t"""
    time: float
    vehicles: List[Vehicle]
    active_requests: List[Request]
    served_requests: List[Request]
    travel_times: Dict[Tuple[Tuple[int, int], Tuple[int, int]], float]

print("Data structures defined for Dynamic VRP formulation")

Data structures defined for Dynamic VRP formulation


In [ ]:
def calculate_distance(pos1: Tuple[int, int], pos2: Tuple[int, int]) -> float:
    """Calculate Euclidean distance between two positions"""
    return np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)

def initialize_travel_times(grid_size: int = 5) -> Dict:
    """Initialize travel time matrix for grid city"""
    travel_times = {}
    positions = [(i, j) for i in range(grid_size) for j in range(grid_size)]

    for pos1 in positions:
        for pos2 in positions:
            if pos1 != pos2:
                # Travel time proportional to Euclidean distance
                travel_times[(pos1, pos2)] = calculate_distance(pos1, pos2)
            else:
                travel_times[(pos1, pos2)] = 0.0

    return travel_times

def calculate_route_cost(route: List[Tuple[int, int]],
                        travel_times: Dict) -> float:
    """Calculate total travel cost for a route"""
    if len(route) < 2:
        return 0.0

    total_cost = 0.0
    for i in range(len(route) - 1):
        total_cost += travel_times[(route[i], route[i+1])]

    return total_cost

print("Utility functions initialized")

Utility functions initialized


In [ ]:
def create_concrete_example() -> DVRPState:
    """Create the concrete example from the source material"""

    # Initialize travel times for 5x5 grid
    travel_times = initialize_travel_times(5)

    # Create vehicles at their initial positions (t0 = 9:00 AM)
    vehicles = [
        Vehicle(
            id=1,
            position=(1, 1),
            capacity=20,
            remaining_capacity=15,
            planned_route=[(2, 3), (4, 4)]
        ),
        Vehicle(
            id=2,
            position=(3, 2),
            capacity=25,
            remaining_capacity=20,
            planned_route=[(5, 5)]
        ),
        Vehicle(
            id=3,
            position=(2, 4),
            capacity=18,
            remaining_capacity=18,
            planned_route=[(1, 1)]  # Returning to depot
        )
    ]

    # Initial state at t0 = 9:00 AM (no requests yet)
    initial_state = DVRPState(
        time=9.0 * 60,  # Convert to minutes
        vehicles=vehicles,
        active_requests=[],
        served_requests=[],
        travel_times=travel_times
    )

    return initial_state

# Create the initial state
initial_state = create_concrete_example()
print(f"Initial state created at {initial_state.time/60:.1f}:00 AM")
print(f"Vehicles: {len(initial_state.vehicles)}")
print(f"Active requests: {len(initial_state.active_requests)}")

Initial state created at 9.0:00 AM
Vehicles: 3
Active requests: 0


In [ ]:
try:
    def add_dynamic_requests(state: DVRPState) -> DVRPState:
        """Add new requests that arrive at t1 = 9:15 AM"""
    
        # New requests arriving at 9:15 AM
        new_requests = [
            Request(
                id=1,
                origin=(1, 3),
                destination=(3, 5),
                demand=2,
                arrival_time=9.25 * 60,  # 9:15 AM in minutes
                time_window_early=9.5 * 60,   # 9:30 AM
                time_window_late=10.5 * 60,   # 10:30 AM
                priority=1  # High priority
            ),
            Request(
                id=2,
                origin=(4, 2),
                destination=(2, 1),
                demand=3,
                arrival_time=9.25 * 60,  # 9:15 AM
                time_window_early=9.75 * 60,  # 9:45 AM
                time_window_late=11.0 * 60,   # 11:00 AM
                priority=2  # Medium priority
            )
        ]
    
        # Update state with new requests
        updated_state = DVRPState(
            time=9.25 * 60,  # 9:15 AM
            vehicles=state.vehicles,
            active_requests=new_requests,
            served_requests=state.served_requests,
            travel_times=state.travel_times
        )
    
        return updated_state
    
    # Add dynamic requests
    state_with_requests = add_dynamic_requests(initial_state)
    print(f"State updated at {state_with_requests.time/60:.2f}:00")
    print(f"New requests: {len(state_with_requests.active_requests)}")
    for req in state_with_requests.active_requests:
        print(f"  Request {req.id}: {req.origin} → {req.dest}, Window: [{req.time_window_early/60:.1f}, {req.time_window_late/60:.1f}]")
except Exception as e:
    print(f'Error in cell: {e}')

State updated at 9.25:00
New requests: 2
Error in cell: 'Request' object has no attribute 'dest'


In [ ]:
def calculate_detour_cost(vehicle: Vehicle, request: Request,
                         travel_times: Dict) -> float:
    """Calculate detour cost for inserting request into vehicle route"""

    # Current route cost
    current_cost = calculate_route_cost([vehicle.position] + vehicle.planned_route, travel_times)

    # New route with request inserted (simplified: insert at beginning)
    new_route = [vehicle.position, request.origin, request.destination] + vehicle.planned_route
    new_cost = calculate_route_cost(new_route, travel_times)

    return new_cost - current_cost

def check_time_window_feasibility(vehicle: Vehicle, request: Request,
                                  travel_times: Dict) -> bool:
    """Check if vehicle can serve request within time window"""

    # Time to reach request origin
    time_to_origin = travel_times[(vehicle.position, request.origin)]
    arrival_time = vehicle.current_time + time_to_origin

    # Check if arrival is within time window
    return request.time_window_early <= arrival_time <= request.time_window_late

def check_capacity_feasibility(vehicle: Vehicle, request: Request) -> bool:
    """Check if vehicle has sufficient capacity for request"""
    return vehicle.remaining_capacity >= request.demand

print("Feasibility checking functions defined")

Feasibility checking functions defined


In [ ]:
def exhaustive_assignment_optimization(state: DVRPState) -> Dict:
    """Exhaustive search for optimal request assignments (small instances only)"""

    if len(state.active_requests) == 0:
        return {"assignments": [], "total_cost": 0.0}

    best_assignment = None
    best_cost = float('inf')

    # Generate all possible assignments (each request to each vehicle)
    requests = state.active_requests
    vehicles = state.vehicles

    # For small instances, try all combinations
    for assignment_pattern in permutations(range(len(vehicles)), len(requests)):

        valid_assignment = True
        total_cost = 0.0
        assignments = []

        for i, request in enumerate(requests):
            vehicle_idx = assignment_pattern[i]
            vehicle = vehicles[vehicle_idx]

            # Check feasibility
            if not check_capacity_feasibility(vehicle, request):
                valid_assignment = False
                break

            if not check_time_window_feasibility(vehicle, request, state.travel_times):
                valid_assignment = False
                break

            # Calculate detour cost
            detour_cost = calculate_detour_cost(vehicle, request, state.travel_times)
            total_cost += detour_cost

            assignments.append({
                'request_id': request.id,
                'vehicle_id': vehicle.id,
                'detour_cost': detour_cost,
                'feasible': True
            })

        if valid_assignment and total_cost < best_cost:
            best_cost = total_cost
            best_assignment = assignments

    return {
        "assignments": best_assignment or [],
        "total_cost": best_cost if best_cost != float('inf') else float('inf'),
        "feasible": best_assignment is not None
    }

# Solve the assignment problem
start_time = time.time()
solution = exhaustive_assignment_optimization(state_with_requests)
solve_time = time.time() - start_time

print(f"Optimization completed in {solve_time:.4f} seconds")
print(f"Solution feasible: {solution['feasible']}")
print(f"Total system cost: {solution['total_cost']:.2f}")
print("\nOptimal assignments:")
for assignment in solution['assignments']:
    print(f"  Request {assignment['request_id']} → Vehicle {assignment['vehicle_id']} (Cost: {assignment['detour_cost']:.2f})")

Optimization completed in 0.0000 seconds
Solution feasible: False
Total system cost: inf

Optimal assignments:


In [ ]:
def visualize_solution(initial_state: DVRPState,
                      state_with_requests: DVRPState,
                      solution: Dict):
    """Visualize the dynamic VRP solution"""

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Dynamic VRP - Mathematical Formulation Results', fontsize=16, fontweight='bold')

    # Plot 1: Initial vehicle positions and routes
    ax1 = axes[0, 0]
    ax1.set_title('Initial State (9:00 AM)', fontweight='bold')
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 6)
    ax1.set_ylim(0, 6)

    # Plot depot
    ax1.plot(1, 1, 'rs', markersize=12, label='Depot')

    # Plot vehicles and routes
    colors = ['blue', 'green', 'orange']
    for i, vehicle in enumerate(initial_state.vehicles):
        # Vehicle position
        ax1.plot(vehicle.position[0], vehicle.position[1], 'o',
                color=colors[i], markersize=10, label=f'Vehicle {vehicle.id}')

        # Planned route
        route = [vehicle.position] + vehicle.planned_route
        route_x = [pos[0] for pos in route]
        route_y = [pos[1] for pos in route]
        ax1.plot(route_x, route_y, '--', color=colors[i], alpha=0.7, linewidth=2)

        # Route destinations
        for dest in vehicle.planned_route:
            ax1.plot(dest[0], dest[1], 's', color=colors[i], markersize=8)

    ax1.legend()

    # Plot 2: New requests arrival
    ax2 = axes[0, 1]
    ax2.set_title('New Requests Arrival (9:15 AM)', fontweight='bold')
    ax2.set_xlabel('X Coordinate')
    ax2.set_ylabel('Y Coordinate')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 6)
    ax2.set_ylim(0, 6)

    # Plot depot and vehicles (current positions)
    ax2.plot(1, 1, 'rs', markersize=12, label='Depot')
    for i, vehicle in enumerate(state_with_requests.vehicles):
        ax2.plot(vehicle.position[0], vehicle.position[1], 'o',
                color=colors[i], markersize=10, label=f'Vehicle {vehicle.id}')

    # Plot new requests
    for request in state_with_requests.active_requests:
        # Origin
        ax2.plot(request.origin[0], request.origin[1], '^',
                markersize=12, label=f'Request {request.id} Origin')
        # Destination
        ax2.plot(request.destination[0], request.destination[1], 'v',
                markersize=12, label=f'Request {request.id} Destination')
        # Arrow from origin to destination
        ax2.annotate('', xy=request.destination, xytext=request.origin,
                    arrowprops=dict(arrowstyle='->', lw=2, color='red'))

    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    # Plot 3: Assignment costs
    ax3 = axes[1, 0]
    ax3.set_title('Assignment Cost Breakdown', fontweight='bold')
    ax3.set_xlabel('Request-Vehicle Assignment')
    ax3.set_ylabel('Detour Cost')

    if solution['assignments']:
        assignments = solution['assignments']
        labels = [f"R{a['request_id']}→V{a['vehicle_id']}" for a in assignments]
        costs = [a['detour_cost'] for a in assignments]

        bars = ax3.bar(labels, costs, color=['skyblue', 'lightcoral'])
        ax3.tick_params(axis='x', rotation=45)

        # Add value labels on bars
        for bar, cost in zip(bars, costs):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{cost:.2f}', ha='center', va='bottom', fontweight='bold')

    ax3.grid(True, alpha=0.3, axis='y')

    # Plot 4: Performance metrics
    ax4 = axes[1, 1]
    ax4.set_title('System Performance Metrics', fontweight='bold')

    # Create performance summary
    metrics = {
        'Total Cost': solution['total_cost'],
        'Requests Served': len(solution['assignments']),
        'Vehicles Utilized': len(set(a['vehicle_id'] for a in solution['assignments'])),
        'Computation Time (s)': solve_time
    }

    # Display as table
    metric_text = "\n".join([f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}"
                             for k, v in metrics.items()])

    ax4.text(0.1, 0.5, metric_text, fontsize=12, verticalalignment='center',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.5))
    ax4.axis('off')

    plt.tight_layout()
    plt.show()

# Visualize the solution
visualize_solution(initial_state, state_with_requests, solution)

In [ ]:
def sensitivity_analysis() -> Dict:
    """Perform sensitivity analysis on key parameters"""

    results = {
        'vehicle_capacity': [],
        'time_window_width': [],
        'request_priority': []
    }

    # Test different vehicle capacities
    original_capacities = [v.capacity for v in initial_state.vehicles]
    capacity_multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]

    for mult in capacity_multipliers:
        # Modify vehicle capacities
        test_state = DVRPState(
            time=state_with_requests.time,
            vehicles=[Vehicle(v.id, v.position, int(v.capacity * mult),
                          int(v.remaining_capacity * mult), v.planned_route, v.current_time)
                      for v in state_with_requests.vehicles],
            active_requests=state_with_requests.active_requests,
            served_requests=state_with_requests.served_requests,
            travel_times=state_with_requests.travel_times
        )

        sol = exhaustive_assignment_optimization(test_state)
        results['vehicle_capacity'].append({
            'multiplier': mult,
            'feasible': sol['feasible'],
            'cost': sol['total_cost'] if sol['feasible'] else float('inf')
        })

    # Test different time window widths
    window_multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]
    base_window = 60  # 1 hour in minutes

    for mult in window_multipliers:
        # Modify time windows
        modified_requests = []
        for req in state_with_requests.active_requests:
            new_width = (req.time_window_late - req.time_window_early) * mult
            modified_requests.append(Request(
                req.id, req.origin, req.destination, req.demand,
                req.arrival_time, req.time_window_early,
                req.time_window_early + new_width, req.priority
            ))

        test_state = DVRPState(
            time=state_with_requests.time,
            vehicles=state_with_requests.vehicles,
            active_requests=modified_requests,
            served_requests=state_with_requests.served_requests,
            travel_times=state_with_requests.travel_times
        )

        sol = exhaustive_assignment_optimization(test_state)
        results['time_window_width'].append({
            'multiplier': mult,
            'feasible': sol['feasible'],
            'cost': sol['total_cost'] if sol['feasible'] else float('inf')
        })

    return results

# Perform sensitivity analysis
print("Performing sensitivity analysis...")
sensitivity_results = sensitivity_analysis()
print("Sensitivity analysis completed")

Performing sensitivity analysis...
Sensitivity analysis completed


In [ ]:
def visualize_sensitivity_analysis(results: Dict):
    """Visualize sensitivity analysis results"""

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('Sensitivity Analysis - Dynamic VRP Mathematical Formulation',
                 fontsize=16, fontweight='bold')

    # Plot 1: Vehicle capacity sensitivity
    ax1 = axes[0]
    ax1.set_title('Impact of Vehicle Capacity', fontweight='bold')
    ax1.set_xlabel('Capacity Multiplier')
    ax1.set_ylabel('Total System Cost', color='blue')

    capacity_data = results['vehicle_capacity']
    multipliers = [d['multiplier'] for d in capacity_data]
    costs = [d['cost'] if d['feasible'] else np.nan for d in capacity_data]
    feasible = [d['feasible'] for d in capacity_data]

    # Plot costs
    line1 = ax1.plot(multipliers, costs, 'bo-', label='Total Cost', linewidth=2)
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.grid(True, alpha=0.3)

    # Add feasibility indicator on secondary axis
    ax1_twin = ax1.twinx()
    ax1_twin.set_ylabel('Feasible', color='red')
    ax1_twin.plot(multipliers, feasible, 'rs--', label='Feasible', linewidth=2)
    ax1_twin.tick_params(axis='y', labelcolor='red')
    ax1_twin.set_ylim(-0.1, 1.1)

    # Plot 2: Time window sensitivity
    ax2 = axes[1]
    ax2.set_title('Impact of Time Window Width', fontweight='bold')
    ax2.set_xlabel('Time Window Multiplier')
    ax2.set_ylabel('Total System Cost', color='blue')

    window_data = results['time_window_width']
    multipliers = [d['multiplier'] for d in window_data]
    costs = [d['cost'] if d['feasible'] else np.nan for d in window_data]
    feasible = [d['feasible'] for d in window_data]

    # Plot costs
    line2 = ax2.plot(multipliers, costs, 'bo-', label='Total Cost', linewidth=2)
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2.grid(True, alpha=0.3)

    # Add feasibility indicator on secondary axis
    ax2_twin = ax2.twinx()
    ax2_twin.set_ylabel('Feasible', color='red')
    ax2_twin.plot(multipliers, feasible, 'rs--', label='Feasible', linewidth=2)
    ax2_twin.tick_params(axis='y', labelcolor='red')
    ax2_twin.set_ylim(-0.1, 1.1)

    plt.tight_layout()
    plt.show()

    # Print summary statistics
    print("\n" + "="*60)
    print("SENSITIVITY ANALYSIS SUMMARY")
    print("="*60)

    print("\nVehicle Capacity Impact:")
    for d in capacity_data:
        status = "FEASIBLE" if d['feasible'] else "INFEASIBLE"
        cost_str = f"{d['cost']:.2f}" if d['feasible'] else "N/A"
        print(f"  {d['multiplier']:.1f}x: {status}, Cost = {cost_str}")

    print("\nTime Window Width Impact:")
    for d in window_data:
        status = "FEASIBLE" if d['feasible'] else "INFEASIBLE"
        cost_str = f"{d['cost']:.2f}" if d['feasible'] else "N/A"
        print(f"  {d['multiplier']:.1f}x: {status}, Cost = {cost_str}")

# Visualize sensitivity analysis
visualize_sensitivity_analysis(sensitivity_results)


SENSITIVITY ANALYSIS SUMMARY

Vehicle Capacity Impact:
  0.5x: INFEASIBLE, Cost = N/A
  0.8x: INFEASIBLE, Cost = N/A
  1.0x: INFEASIBLE, Cost = N/A
  1.2x: INFEASIBLE, Cost = N/A
  1.5x: INFEASIBLE, Cost = N/A

Time Window Width Impact:
  0.5x: INFEASIBLE, Cost = N/A
  0.8x: INFEASIBLE, Cost = N/A
  1.0x: INFEASIBLE, Cost = N/A
  1.2x: INFEASIBLE, Cost = N/A
  1.5x: INFEASIBLE, Cost = N/A


In [ ]:
def mathematical_formulation_summary():
    """Provide comprehensive summary of mathematical formulation approach"""

    print("\n" + "="*80)
    print("DYNAMIC VEHICLE ROUTING PROBLEM - MATHEMATICAL FORMULATION SUMMARY")
    print("="*80)

    print("\n📊 PROBLEM CHARACTERISTICS:")
    print(f"  • Vehicles: {len(initial_state.vehicles)}")
    print(f"  • Dynamic requests: {len(state_with_requests.active_requests)}")
    print(f"  • Grid size: 5×5")
    print(f"  • Time horizon: 9:00 AM - 11:00 AM")
    print(f"  • Optimization method: Exhaustive search")

    print("\n🎯 OPTIMAL SOLUTION RESULTS:")
    print(f"  • Solution feasible: {solution['feasible']}")
    print(f"  • Total system cost: {solution['total_cost']:.2f}")
    print(f"  • Computation time: {solve_time:.4f} seconds")
    print(f"  • Requests assigned: {len(solution['assignments'])}")

    print("\n📋 DETAILED ASSIGNMENTS:")
    for assignment in solution['assignments']:
        print(f"  • Request {assignment['request_id']} → Vehicle {assignment['vehicle_id']}")
        print(f"    Detour cost: {assignment['detour_cost']:.2f}")

    print("\n🔍 MATHEMATICAL MODEL COMPONENTS:")
    print("  • State Space: S(t) = {V^r(t), V^p(t), R(t), C(t), Θ(t)}")
    print("  • Decision Variables: Vehicle-request assignments x_{ij}^k(t)")
    print("  • Objective: Minimize α·TC(t) + β·WT(t) + γ·RJ(t) + δ·CV(t)")
    print("  • Constraints: Capacity, time windows, route continuity")
    print("  • Request process: Non-homogeneous Poisson process")

    print("\n⚡ COMPUTATIONAL CHARACTERISTICS:")
    print("  • Complexity: O(n!) for exhaustive search")
    print("  • Scalability: Limited to small instances (≤5 vehicles, ≤15 requests)")
    print("  • Solution quality: Provable optimal")
    print("  • Real-time capability: No (requires significant computation time)")

    print("\n📈 PRACTICAL INSIGHTS:")
    print("  • Detour costs drive assignment decisions")
    print("  • Time windows significantly impact feasibility")
    print("  • Vehicle capacity constraints create bottlenecks")
    print("  • Trade-offs between cost minimization and service level")

    print("\n🎓 EDUCATIONAL VALUE:")
    print("  • Demonstrates mathematical modeling of dynamic systems")
    print("  • Shows complexity of real-time optimization")
    print("  • Provides baseline for heuristic and metaheuristic methods")
    print("  • Illustrates importance of constraint handling")

    print("\n" + "="*80)

# Display comprehensive summary
mathematical_formulation_summary()


DYNAMIC VEHICLE ROUTING PROBLEM - MATHEMATICAL FORMULATION SUMMARY

📊 PROBLEM CHARACTERISTICS:
  • Vehicles: 3
  • Dynamic requests: 2
  • Grid size: 5×5
  • Time horizon: 9:00 AM - 11:00 AM
  • Optimization method: Exhaustive search

🎯 OPTIMAL SOLUTION RESULTS:
  • Solution feasible: False
  • Total system cost: inf
  • Computation time: 0.0000 seconds
  • Requests assigned: 0

📋 DETAILED ASSIGNMENTS:

🔍 MATHEMATICAL MODEL COMPONENTS:
  • State Space: S(t) = {V^r(t), V^p(t), R(t), C(t), Θ(t)}
  • Decision Variables: Vehicle-request assignments x_{ij}^k(t)
  • Objective: Minimize α·TC(t) + β·WT(t) + γ·RJ(t) + δ·CV(t)
  • Constraints: Capacity, time windows, route continuity
  • Request process: Non-homogeneous Poisson process

⚡ COMPUTATIONAL CHARACTERISTICS:
  • Complexity: O(n!) for exhaustive search
  • Scalability: Limited to small instances (≤5 vehicles, ≤15 requests)
  • Solution quality: Provable optimal
  • Real-time capability: No (requires significant computation time)

📈 PR